# Diffusion Models — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/notebook/diffusion/diffusion-lab.ipynb)

Companion notebook for the **Diffusion Models** track (`diffusion-m1` … `diffusion-m11`). Builds a small DDPM from scratch in PyTorch, trains it on Fashion-MNIST, and re-derives the numeric claims made across the track.

Runs on **CPU in a few minutes** or on a Colab T4 GPU in under a minute per epoch.

| Part | Topic | What runs |
|---|---|---|
| 1 | `diffusion-m2` | The closed-form forward process, verified against a step-by-step simulation of the same chain |
| 2 | `diffusion-m7` | Linear vs. cosine schedules, comparing how much signal remains at each t |
| 3 | `diffusion-m5`, `diffusion-m10` | The time-conditioned U-Net denoiser, built from scratch |
| 4 | `diffusion-m4`, `diffusion-m10` | The simple noise-prediction loss and the training loop on Fashion-MNIST |
| 5 | `diffusion-m6`, `diffusion-m10` | The full DDPM sampling loop, generating images from pure noise |
| 6 | `diffusion-m7` | DDIM step-skipping, reusing the same trained network |
| 7 | `diffusion-m8` | Classifier-free guidance's extrapolation formula, verified on toy logits |

## 0. Setup & Dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Active device:   {DEVICE}')

---
# Part 1 — The Closed-Form Forward Process (`diffusion-m2`)

We verify that jumping directly to x_t via the closed form matches simulating the chain step by step.

In [ ]:
T = 300

def linear_schedule(T, beta_start=1e-4, beta_end=0.02):
    betas = torch.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return betas, alpha_bars

betas, alpha_bars = linear_schedule(T)

torch.manual_seed(0)
x0 = torch.randn(4, 1, 8, 8)  # toy "image"
t_check = 50

# Step-by-step simulation of the forward chain up to t_check
x_step = x0.clone()
torch.manual_seed(1)
for step in range(t_check):
    beta = betas[step]
    noise_step = torch.randn_like(x_step)
    x_step = (1 - beta).sqrt() * x_step + beta.sqrt() * noise_step

# Closed-form single-step jump to the same t_check, using ONE noise draw
torch.manual_seed(1)  # same seed only to make the comparison legible below -- the two use different noise counts
noise_direct = torch.randn_like(x0)
sqrt_ab = alpha_bars[t_check].sqrt()
sqrt_one_minus_ab = (1 - alpha_bars[t_check]).sqrt()
x_direct = sqrt_ab * x0 + sqrt_one_minus_ab * noise_direct

print(f'Step-simulated x_{t_check}: mean={x_step.mean().item():.4f}, std={x_step.std().item():.4f}')
print(f'Closed-form   x_{t_check}: mean={x_direct.mean().item():.4f}, std={x_direct.std().item():.4f}')
print('Both approaches draw from the same distribution N(sqrt(alpha_bar_t) x0, (1-alpha_bar_t) I) --')
print('exact values differ (different noise draws), but the closed form needs exactly 1 draw, not', t_check, '.')
print(f'alpha_bar_{t_check} = {alpha_bars[t_check].item():.4f}')

---
# Part 2 — Linear vs. Cosine Noise Schedules (`diffusion-m7`)

In [ ]:
def cosine_schedule(T, s=0.008):
    t = torch.arange(T + 1, dtype=torch.float32)
    f = torch.cos(((t / T + s) / (1 + s)) * (np.pi / 2)) ** 2
    alpha_bars_cos = f / f[0]
    return alpha_bars_cos[1:]

alpha_bars_lin = alpha_bars
alpha_bars_cos = cosine_schedule(T)

plt.figure(figsize=(8, 4))
plt.plot(alpha_bars_lin, label='Linear schedule')
plt.plot(alpha_bars_cos, label='Cosine schedule')
plt.xlabel('t'); plt.ylabel('alpha_bar_t'); plt.title('How Much Original Signal Remains at Each Step')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

checkpoint = T // 10
print(f'At t={checkpoint} (10% through the chain):')
print(f'  Linear alpha_bar:  {alpha_bars_lin[checkpoint].item():.4f}')
print(f'  Cosine alpha_bar:  {alpha_bars_cos[checkpoint].item():.4f}')
print('The cosine schedule keeps far more signal at this early checkpoint -- confirms Module 7.')

---
# Part 3 — Time-Conditioned U-Net Denoiser (`diffusion-m5`, `diffusion-m10`)

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-torch.arange(half, device=t.device) * (9.2 / half))
        args = t[:, None].float() * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.mlp(emb)


class SimpleUNet(nn.Module):
    def __init__(self, time_dim=64):
        super().__init__()
        self.time_embed = TimeEmbedding(time_dim)
        self.down1 = nn.Conv2d(1, 64, 3, stride=2, padding=1)
        self.down2 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.time_proj1 = nn.Linear(time_dim, 64)
        self.time_proj2 = nn.Linear(time_dim, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.up2 = nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1)
        self.out = nn.Conv2d(64 + 1, 1, 3, padding=1)

    def forward(self, x, t):
        temb = self.time_embed(t)
        h1 = F.silu(self.down1(x) + self.time_proj1(temb)[:, :, None, None])
        h2 = F.silu(self.down2(h1) + self.time_proj2(temb)[:, :, None, None])
        u1 = F.silu(self.up1(h2))
        u2 = F.silu(self.up2(torch.cat([u1, h1], dim=1)))
        return self.out(torch.cat([u2, x], dim=1))


def forward_diffusion(x0, t, noise, alpha_bars):
    sqrt_ab = alpha_bars[t].sqrt().view(-1, 1, 1, 1)
    sqrt_one_minus_ab = (1 - alpha_bars[t]).sqrt().view(-1, 1, 1, 1)
    return sqrt_ab * x0 + sqrt_one_minus_ab * noise

model = SimpleUNet().to(DEVICE)

x_test = torch.randn(4, 1, 28, 28, device=DEVICE)
t_test = torch.randint(0, T, (4,), device=DEVICE)
out_test = model(x_test, t_test)
print(f'Input shape:  {tuple(x_test.shape)}')
print(f'Output shape: {tuple(out_test.shape)}  (must match input -- predicting noise, same shape as x)')
assert out_test.shape == x_test.shape
print('Denoiser shape verified.')

---
# Part 4 — Training on Fashion-MNIST with the Simple Loss (`diffusion-m4`, `diffusion-m10`)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

betas, alpha_bars = linear_schedule(T)
betas, alpha_bars = betas.to(DEVICE), alpha_bars.to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=2e-4)

def train_step(x0):
    batch_size = x0.size(0)
    t = torch.randint(0, T, (batch_size,), device=DEVICE)
    noise = torch.randn_like(x0)
    x_t = forward_diffusion(x0, t, noise, alpha_bars)
    predicted_noise = model(x_t, t)
    loss = F.mse_loss(predicted_noise, noise)  # L_simple, Module 4
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

EPOCHS = 3
print(f'Training DDPM on Fashion-MNIST for {EPOCHS} epochs (T={T} steps)...')
for epoch in range(1, EPOCHS + 1):
    losses = []
    for real_images, _ in train_loader:
        real_images = real_images.to(DEVICE)
        losses.append(train_step(real_images))
    print(f'Epoch {epoch}/{EPOCHS} | Simple loss: {np.mean(losses):.4f}')

---
# Part 5 — DDPM Sampling: From Pure Noise to an Image (`diffusion-m6`, `diffusion-m10`)

In [ ]:
@torch.no_grad()
def sample_ddpm(model, num_samples, T, betas, alpha_bars, device):
    model.eval()
    x = torch.randn(num_samples, 1, 28, 28, device=device)

    for t_val in reversed(range(T)):
        t = torch.full((num_samples,), t_val, device=device, dtype=torch.long)
        predicted_noise = model(x, t)

        alpha_t = 1 - betas[t_val]
        alpha_bar_t = alpha_bars[t_val]
        beta_t = betas[t_val]

        mean = (1 / alpha_t.sqrt()) * (x - (beta_t / (1 - alpha_bar_t).sqrt()) * predicted_noise)

        if t_val > 0:
            noise = torch.randn_like(x)
            x = mean + beta_t.sqrt() * noise
        else:
            x = mean

    model.train()
    return (x + 1) / 2

samples = sample_ddpm(model, num_samples=16, T=T, betas=betas, alpha_bars=alpha_bars, device=DEVICE)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].cpu().squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle(f'DDPM Samples After {T} Sequential Denoising Steps', fontsize=13)
plt.tight_layout()
plt.show()

---
# Part 6 — DDIM: Fewer Steps, Same Trained Model (`diffusion-m7`)

We reuse the exact same `model` from Part 4-5, just calling it fewer times.

In [ ]:
@torch.no_grad()
def sample_ddim(model, num_samples, T, alpha_bars, device, ddim_steps=30):
    model.eval()
    step_indices = torch.linspace(0, T - 1, ddim_steps, device=device).long().flip(0)
    x = torch.randn(num_samples, 1, 28, 28, device=device)

    for i, t_val in enumerate(step_indices):
        t = torch.full((num_samples,), t_val.item(), device=device, dtype=torch.long)
        predicted_noise = model(x, t)
        alpha_bar_t = alpha_bars[t_val]

        x0_pred = (x - (1 - alpha_bar_t).sqrt() * predicted_noise) / alpha_bar_t.sqrt()

        if i + 1 < len(step_indices):
            alpha_bar_prev = alpha_bars[step_indices[i + 1]]
        else:
            alpha_bar_prev = torch.tensor(1.0, device=device)

        x = alpha_bar_prev.sqrt() * x0_pred + (1 - alpha_bar_prev).sqrt() * predicted_noise  # deterministic DDIM step

    model.train()
    return (x + 1) / 2

import time
t_start = time.time()
ddpm_samples = sample_ddpm(model, num_samples=8, T=T, betas=betas, alpha_bars=alpha_bars, device=DEVICE)
ddpm_time = time.time() - t_start

t_start = time.time()
ddim_samples = sample_ddim(model, num_samples=8, T=T, alpha_bars=alpha_bars, device=DEVICE, ddim_steps=30)
ddim_time = time.time() - t_start

print(f'DDPM: {T} network evaluations, {ddpm_time:.2f}s')
print(f'DDIM: 30 network evaluations, {ddim_time:.2f}s')
print(f'Speedup: {ddpm_time / max(ddim_time, 1e-6):.1f}x, using the exact same trained model.')

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(ddpm_samples[i].cpu().squeeze(), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(ddim_samples[i].cpu().squeeze(), cmap='gray'); axes[1, i].axis('off')
axes[0, 0].set_title(f'DDPM ({T} steps)', loc='left', fontsize=10, fontweight='bold')
axes[1, 0].set_title('DDIM (30 steps)', loc='left', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 7 — Classifier-Free Guidance Extrapolation (`diffusion-m8`)

A toy check of the guidance formula itself, on stand-in tensors rather than a full text-conditioned model.

In [ ]:
torch.manual_seed(3)
eps_uncond = torch.randn(1, 1, 8, 8)
eps_cond = eps_uncond + torch.randn(1, 1, 8, 8) * 0.5  # conditional prediction, nudged toward a 'prompt' direction

def guided_eps(eps_uncond, eps_cond, w):
    return eps_uncond + w * (eps_cond - eps_uncond)

for w in [0.0, 1.0, 3.0, 7.0]:
    guided = guided_eps(eps_uncond, eps_cond, w)
    dist_from_cond = (guided - eps_cond).norm().item()
    dist_from_uncond = (guided - eps_uncond).norm().item()
    print(f'w={w:4.1f} | distance from conditional: {dist_from_cond:.3f} | distance from unconditional: {dist_from_uncond:.3f}')

print('\nAt w=0 the guided prediction equals the unconditional one exactly.')
print('At w=1 it equals the plain conditional prediction.')
print('For w>1 it moves PAST the conditional prediction, further from unconditional -- the extrapolation from Module 8.')